# Count annotated phenotypes

[Standard notebook](https://github.com/EinarOlafsson/spacr/blob/nightly/Notebooks/19_count_phenotypes.ipynb) · [Help](https://github.com/EinarOlafsson/spacr/blob/nightly/Notebooks/reference/19_count_phenotypes.md) · [All workflows](https://github.com/EinarOlafsson/spacr/blob/nightly/Notebooks/README.md)

[Setup](#setup) · [Data](#data) · [Inputs](#inputs) · [Advanced](#advanced) · [Run](#run) · [Outputs](#outputs)

## Setup

In [ ]:
#@title 1. Setup (check Install only when needed)
INSTALL_SPACR = False #@param {"type":"boolean"}
PACKAGE = "spacr" #@param {"type":"string"}
if INSTALL_SPACR:
    import subprocess, sys
    if PACKAGE != "spacr" and not (PACKAGE.startswith("spacr==") and all(c.isdigit() or c == "." for c in PACKAGE[7:])):
        raise ValueError("Use spacr or spacr==<published version>.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", PACKAGE, "ipywidgets"])
from importlib.metadata import version
print("spaCR", version("spacr"))
from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
#@title 2. Load matching notebook controls
from pathlib import Path
from urllib.request import urlopen
import hashlib
helper = Path("_spacr_notebook_widgets.py")
expected = 'b3cee160648607b49ef051c2424c06a28631a4a11a7b52d1063f73cec643f7c0'
if not helper.is_file() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
    payload = urlopen("https://raw.githubusercontent.com/EinarOlafsson/spacr/nightly/Notebooks/_spacr_notebook_widgets.py", timeout=30).read()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise RuntimeError("Notebook/helper versions differ. Reopen the latest notebook before continuing.")
    helper.write_bytes(payload)


## Data

In [ ]:
#@title 3. Data (Drive/upload only when selected)
DATA_SOURCE = "Existing runtime path" #@param ["Existing runtime path", "Google Drive", "Upload files"]
DATA_PATH = "/content/data" #@param {"type":"string"}
if DATA_SOURCE == "Google Drive":
    from google.colab import drive
    drive.mount("/content/drive")
elif DATA_SOURCE == "Upload files":
    from google.colab import files
    from pathlib import Path
    destination = Path(DATA_PATH)
    destination.mkdir(parents=True, exist_ok=True)
    for filename, payload in files.upload().items():
        (destination / Path(filename).name).write_bytes(payload)
print("Set workflow paths below to your data location:", DATA_PATH)


## Inputs

In [ ]:
#@title 4. Load workflow
from spacr.submodules import count_phenotypes

In [ ]:
#@title 5. Load current defaults
import ast, inspect, textwrap
from typing import Any, Dict
def keys_from_source(func) -> Dict[str, Any]:
    """Keys the function reads off its ``settings`` argument.

    Walks the AST rather than the text, so a key inside a comment or a
    string is not mistaken for one that is read.
    """
    try:
        source = textwrap.dedent(inspect.getsource(func))
        tree = ast.parse(source)
    except (OSError, TypeError, SyntaxError):
        return {}
    found: Dict[str, Any] = {}
    for node in ast.walk(tree):
        if (isinstance(node, ast.Call)
                and isinstance(node.func, ast.Attribute)
                and node.func.attr in ("get", "setdefault")
                and isinstance(node.func.value, ast.Name)
                and node.func.value.id == "settings"
                and node.args
                and isinstance(node.args[0], ast.Constant)
                and isinstance(node.args[0].value, str)):
            default: Any = None
            if len(node.args) > 1:
                try:
                    default = ast.literal_eval(node.args[1])
                except (ValueError, SyntaxError):
                    default = None
            found.setdefault(node.args[0].value, default)
        elif (isinstance(node, ast.Subscript)
                and isinstance(node.value, ast.Name)
                and node.value.id == "settings"
                and isinstance(node.slice, ast.Constant)
                and isinstance(node.slice.value, str)):
            found.setdefault(node.slice.value, None)
    return found
from spacr.submodules import count_phenotypes as _function
settings = keys_from_source(_function)
import copy
_defaults_settings = copy.deepcopy(settings)

In [ ]:
#@title 6.1 Essential inputs — count_phenotypes
stage_1_src = '/content/data' #@param {"type": "string"}
settings.update({'src': stage_1_src})

## Advanced

In [ ]:
#@title 7. Optional Advanced settings and Help (closed until clicked)
try:
    from _spacr_notebook_widgets import show_advanced_settings
except ImportError:
    show_advanced_settings = None
    print('Optional editor helper unavailable; use the advanced reference and advanced_overrides.')
settings_editor = (show_advanced_settings(settings, ['src'], _defaults_settings)
                   if show_advanced_settings is not None else None)
from _spacr_notebook_widgets import show_help_menu
help_menu = show_help_menu('reference/19_count_phenotypes.md')

## Run

In [ ]:
#@title 8. Run (may write files or download models)
RUN_WORKFLOW = False #@param {"type":"boolean"}
if RUN_WORKFLOW:
    count_phenotypes(settings)
else:
    print("No analysis started. Configure inputs before enabling Run.")

## Outputs

In [ ]:
#@title 9. Inspect outputs (bounded list; preview only on click)
PREVIEW_OUTPUTS = False #@param {"type":"boolean"}
OUTPUT_FOLDER = "/content/data" #@param {"type":"string"}
if PREVIEW_OUTPUTS:
    from _spacr_notebook_widgets import preview_outputs
    output_picker, preview_button, output_display = preview_outputs(OUTPUT_FOLDER)


In [ ]:
#@title 10. Download one selected output (optional)
DOWNLOAD_FILE = False #@param {"type":"boolean"}
FILE_PATH = "" #@param {"type":"string"}
if DOWNLOAD_FILE:
    from pathlib import Path
    path = Path(FILE_PATH).expanduser()
    if not path.is_file():
        raise ValueError("Choose an existing output file.")
    from google.colab import files
    files.download(str(path))
